# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Google Collab setup
### Installations

In [ ]:
!pip install mlflow

### Data location

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
os.listdir('/content/drive/MyDrive/data/Brain Tumor MRI')

['data', 'models']

## General

In [ ]:
import mlflow
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("keras_example")

In [1]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121

In [2]:
# path management
PROJECT_ROOT = '/content/drive/MyDrive/data/Brain Tumor MRI'
RAW_DIR = PROJECT_ROOT + "/data/raw"
PREP_DIR = PROJECT_ROOT + "/data/processed"
ARTEFACTS_DIR = PROJECT_ROOT + "/models"

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

## Modeling

### Backbone

In [12]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = False

print("✅ RadImageNet DenseNet121 loaded successfully")

✅ RadImageNet DenseNet121 loaded successfully


In [9]:
backbone.summary()

Model: "densenet121"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 260, 260,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_6    │ (None, 266, 266,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 130, 130,  │      9,408 │ zero_padding2d_6… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 130, 130,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 130, 130,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_7    │ (None, 132, 132,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 65, 65,    │          0 │ zero_padding2d_7… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 65, 65,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 65, 65,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 65, 65,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 65, 65,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 65, 65,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 65, 65,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 65, 65,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 65, 65,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 65, 65,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 65, 65,    │     12,288 │ conv2_block2_0_r

 Total params: 7,037,504 (26.85 MB)

 Trainable params: 6,953,856 (26.53 MB)

 Non-trainable params: 83,648 (326.75 KB)

In [10]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [13]:
model_shared_part = keras.Sequential([
    #Data augmentation
    #TODO if need
    # Base
    backbone,
    # Head
    layers.GlobalAveragePooling2D(), # to flatten backbone output but with moderate position importance and more stable for MRI
    layers.Dense(512, use_bias=False), # 512 because it half of the backbone output (1024)
    layers.BatchNormalization(), # to normalize weight before heads
    layers.Activation('relu'),
    #layers.Dropout(0.4) # to reduce over-fitting risks
], name='shared_part')

In [15]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    #layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [17]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    #layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [19]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = model_shared_part(inputs)
output1 = model_head1(x)
output2 = model_head2(x)

model = keras.Model(inputs=inputs, outputs=[output1, output2], name='densenet_two_head')

In [22]:
model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        'tumor_presence': 'binary_crossentropy',
        'tumor_type': 'categorical_crossentropy'
    },
    metrics={
        'tumor_presence': ['accuracy'],
        'tumor_type': ['accuracy']
    }
)

In [21]:
model.summary()

Model: "densenet_two_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 260, 260,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_part         │ (None, 512)       │  7,563,840 │ input_layer_7[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tumor_presence      │ (None, 1)         │     66,177 │ shared_part[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tumor_type          │ (None, 4)         │     66,564 │ shared_part[0][0] │
│ (Sequential)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,696,581 (29.36 MB)

 Trainable params: 657,541 (2.51 MB)

 Non-trainable params: 7,039,040 (26.85 MB)

In [ ]:
Warning : do not forget :
converted = tf.image.grayscale_to_rgb(original)